In [ ]:
!pip install langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers huggingface_hub -q

In [ ]:
import os
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

In [ ]:
HF_TOKEN = 'YOUR_HF_TOKEN_HERE'
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN

In [ ]:
documents = [
    'Artificial Intelligence is the simulation of human intelligence processes by machines.',
    'Machine Learning is a subset of AI that enables systems to learn from data automatically.',
    'Deep Learning uses neural networks with many layers to learn complex patterns.',
    'Natural Language Processing allows machines to understand and generate human language.',
    'Computer Vision is a field of AI that enables machines to interpret visual information.',
    'Reinforcement Learning is a type of ML where agents learn by interacting with an environment.',
    'Transfer Learning involves using a pre-trained model on a new but related task.',
    'Large Language Models like GPT and BERT are trained on massive text datasets.',
    'RAG stands for Retrieval-Augmented Generation which combines retrieval with generation.',
    'LangChain is a framework for building applications powered by language models.'
]

print('Loaded', len(documents), 'documents')

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.create_documents(documents)
print('Total chunks:', len(chunks))

In [ ]:
print('Creating embeddings and vector store...')
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
print('Vector store created!')

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id='mistralai/Mistral-7B-Instruct-v0.1',
    huggingfacehub_api_token=HF_TOKEN,
    temperature=0.5,
    max_new_tokens=200
)

memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory
)

print('Chatbot ready!')

In [ ]:
print('=' * 50)
print('   Context-Aware Chatbot (RAG)')
print('=' * 50)
print("Type 'quit' to exit\n")

while True:
    user = input('You: ').strip()
    if not user:
        continue
    if user.lower() == 'quit':
        print('Goodbye!')
        break
    try:
        response = chain({'question': user})
        print('Bot:', response['answer'], '\n')
    except Exception as e:
        print('Error:', str(e), '\n')